## Model I/O

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

### Prompts (프롬프트)
모델에게 던지는 질문을 템플릿화 하는 기술

**주요 사용처**

1. **자동화된 입력 구성**
    - PromptTemplate을 사용하여 사용자 입력을 자동으로 구성.
    - 동일한 형식의 질문이나 대화를 대량으로 생성 가능.

2. **대화형 응답**
    - ChatPromptTemplate을 통해 대화형 AI의 문맥 유지를 지원.

3. **결과 파싱**
    - Output Parser를 통해 LLM의 출력을 특정 포맥으로 처리하여 후속 작업 자동화.

In [2]:
%pip install langchain-core

Note: you may need to restart the kernel to use updated packages.


In [3]:
# PromptTemplate
from langchain_core.prompts import PromptTemplate

# 템플릿 정의
template = "{product}를 홍보하기 위한 재미있고, 새로운 광고문구를 작성해 주세요."

# PromptTemplate 객체 생성
prompt = PromptTemplate(
    template=template,
    input_variables=['product']
)

# 실제 프롬프트 생성 (변수 채우기)
print(prompt.format(product='카메라'))
print(prompt.format(product='자동차'))

c:\Users\Admin\miniconda3\envs\pystudy_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


카메라를 홍보하기 위한 재미있고, 새로운 광고문구를 작성해 주세요.
자동차를 홍보하기 위한 재미있고, 새로운 광고문구를 작성해 주세요.


In [4]:
# ChatPromptTemplate - System, Human, AI 유형별 메시지를 작성할 수 있는 기능 제공
from langchain_core.prompts.chat import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

# 시스템 메세지 (역할 부여)
system_message = SystemMessagePromptTemplate.from_template("당신은 도움을 주는 친절한 챗봇입니다.")

# 휴먼 메세지 (사용자 질문)
human_message = HumanMessagePromptTemplate.from_template("질문 : {question}")

# 챗 프롬프트 합치기
messages = ChatPromptTemplate.from_messages([system_message, human_message])
print(messages)

# 포멧팅 확인
prompt = messages.format_messages(question="AI가 무엇인가요?")
print(prompt)

input_variables=['question'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 도움을 주는 친절한 챗봇입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='질문 : {question}'), additional_kwargs={})]
[SystemMessage(content='당신은 도움을 주는 친절한 챗봇입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='질문 : AI가 무엇인가요?', additional_kwargs={}, response_metadata={})]


In [5]:
# FewShotPromptTemplate - n-shot 프롬프팅을 위한 클래스 (말보다 '예시'로 가르치기)
from langchain_core.prompts.few_shot import FewShotPromptTemplate

# 예시 데이터
examples = [
    {"question": "2 + 2는 무엇인가요?", "answer" : "2 + 2 = 4"},
    {"question": "3 + 5는 무엇인가요?", "answer" : "3 + 5 = 8"}
]

# 예시를 어떻게 보여줄지 포맨 정의
example_prompt = PromptTemplate(
    template="Q: {question}\nA: {answer}",
    input_variables=['question', 'answer']
)

# FewShotPromptTemplate 조립
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,                  # 예시 데이터
    example_prompt=example_prompt,      # 예시 포맷
    prefix="다음 계산 문제를 해결하세요",   # 지시어 (앞부분)
    suffix="Q: {question}\nA:",         # 실제 질문 (뒷부분)
    input_variables=['question']
)

prompt = few_shot_prompt.format(question="35 + 48은 무엇인가요?")
print(prompt)

다음 계산 문제를 해결하세요

Q: 2 + 2는 무엇인가요?
A: 2 + 2 = 4

Q: 3 + 5는 무엇인가요?
A: 3 + 5 = 8

Q: 35 + 48은 무엇인가요?
A:


### Output Parser (출력 파서)

LLM이 생성한 텍스트 출력을 특정 형식으로 변환하거나 처리하는데 사용
- 텍스트 기반 응답을 JSON, 리스트 또는 숫자와 같은 특정 포맷으로 변환하여 후속 작업에 활용

**종류**
1. **BaseOutputParser** : Output Parsers의 기본 클래스, 커스텀 파서 구현 시 사용
2. **CommaSeparatedListOutputParser** : 콤마로 구분된 문자열을 리스트로 변환
3. **RegexParser** : 정규식을 사용해 특정 패턴을 추출하고 키-값 형태로 반환
4. **StructuredOutputParser** : 출력의 JSON 또는 구조화된 형식을 강제
5. **PydanticOutputParser** : pydantic 모델을 기반으로 출력 검증 및 변환
6. **MarkdownOutputParser** : 마크다운 형식의 텍스트에서 데이터를 추출

In [6]:
# %pip install langchain-openai

In [7]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_openai import ChatOpenAI

# 파서 생성
output_parser = CommaSeparatedListOutputParser()

# 지시사항 가져오기
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

# 프롬프트 템플릿에 지시사항 주입
prompt_template = PromptTemplate(
    template = "{subject} 5개의 팀을 보여주세요.\n형식지정: {format_instructions}",
    input_variables = ['subject'],  # 사용자 입력값 변수
    partial_variables={'format_instructions' : format_instructions} # 고정적(기본적으로 포함)으로 설정될 변수
)

# 모델 준비
model = ChatOpenAI(model_name="gpt-5-nano", temperature=0)

query = "한국의 프로야구팀"
prompt = prompt_template.format(subject=query)

# invoke로 실행
response = model.invoke(prompt)
print("원본 응답:", response.content)

# 파싱 (문자열 -> 리스트로 변환)
parsed_result = output_parser.parse(response.content)
print("파싱 결과:", parsed_result)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`
원본 응답: 두산 베어스, LG 트윈스, KT Wiz, 기아 타이거즈, NC 다이노스
파싱 결과: ['두산 베어스', 'LG 트윈스', 'KT Wiz', '기아 타이거즈', 'NC 다이노스']


### Models (다양한 모델 사용하기)

In [8]:
# ChatOpenAI
from langchain_openai import OpenAI

model_gpt = ChatOpenAI(model="gpt-5-nano", temperature=1)
model_gpt.invoke("저는 아침으로 사과를 먹었습니다. 저는 아침에 무엇을 먹었을까요?")

AIMessage(content='정답은: 사과를 먹었습니다.\n\n설명: 주어진 문장에 따르면 당신은 아침에 사과를 먹었으므로, 아침에 먹은 것은 사과입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 947, 'prompt_tokens': 28, 'total_tokens': 975, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 896, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CqBl77KWrEosZXJ4pF1ZOVFTkwKKW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019b4ed7-5f59-7c03-9330-9c3772fefbdb-0', usage_metadata={'input_tokens': 28, 'output_tokens': 947, 'total_tokens': 975, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 896}})

In [9]:
# %pip install langchain-huggingface

In [10]:
# %pip install -U huggingface_hub

In [11]:
# HuggingFaceEndpoint(오픈소스 모델)
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

end_point = HuggingFaceEndpoint(
    repo_id='MLP-KTLim/llama-3-Korean-Bllossom-8B',
    task='text-generation',
    max_new_tokens=1024
)

# 챗 모델로 감싸기
hf_model = ChatHuggingFace(llm=end_point)

hf_model.invoke("저는 아침으로 사과를 먹었습니다. 저는 아침에 무엇을 먹었을까요?")

AIMessage(content='안녕하세요! 아침에 무엇을 먹었는지는 질문자에게 달려 있습니다. 저는 "사과"를 말한 것 같은데요?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 43, 'total_tokens': 75}, 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019b4ed7-7cd6-7201-9bd8-50c4dbbc9e41-0', usage_metadata={'input_tokens': 43, 'output_tokens': 32, 'total_tokens': 75})

### ModelLaboratory(모델 비교)

In [12]:
# %pip install langchain-classic

In [13]:
from langchain_classic.model_laboratory import ModelLaboratory

model_lab = ModelLaboratory.from_llms([model_gpt, hf_model])
model_lab.compare("대한민국의 겨울은 몇 월부터 몇 월인가요??")

Input:
대한민국의 겨울은 몇 월부터 몇 월인가요??

profile={'max_input_tokens': 272000, 'max_output_tokens': 128000, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x00000286EDA81250> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000286EDA82690> root_client=<openai.OpenAI object at 0x00000286D1718980> root_async_client=<openai.AsyncOpenAI object at 0x00000286EDA81100> model_name='gpt-5-nano' temperature=1.0 model_kwargs={} openai_api_key=SecretStr('**********') stream_usage=True
두 가지 기준이 있습니다.

- 기상학적 겨울: 12월 ~ 2월 (즉 12, 1, 2월)
- 천문학적 겨울: 대략 12월 말경 시작(보통 12월 21일 무렵)해서 3월 말경까지이며, 해마다 하루 정도 차이가 있습니다(대략 